## 7a. Linear Regression Model Training and Testing

Code are following AAI 540 Lab 4

### Setup to Load Data from S3

In [1]:
# !pip install --upgrade pip

In [2]:
# !pip install -q PyAthena

In [8]:
import boto3
import sagemaker
import pandas as pd
from pyathena import connect
sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name

In [9]:

# Set S3 path to Parquet data
s3_path_parquet = f's3://{bucket}/CCPP/data//parquet'
# Set Athena parameters
# Set Athena parameters
database_name = "ccpp_aws_fp"
table_name_csv = "final_data_csv"
table_name_parquet = "final_data_parquet_v1"

In [10]:
# Set S3 staging directory -- this is a temporary directory used for Athena queries
s3_staging_dir = "s3://{0}/athena/staging".format(bucket)

In [11]:
conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)

In [ ]:
from pyathena import connect

In [14]:

statement = """SELECT count(1) FROM {}.{}
    """.format(
    database_name, table_name_parquet
)

print(statement) 

SELECT count(1) FROM ccpp_aws_fp.final_data_parquet_v1
    


In [18]:

data_count= pd.read_sql(statement, conn)
total_records=data_count['_col0'].values[0]
total_records

/tmp/ipykernel_20296/3056553487.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_count= pd.read_sql(statement, conn)


14764

In [21]:
train_seq_id= int(total_records*0.4)
val_seq_id= train_seq_id+ int(total_records*0.4)


### Load Training Dataset (all year 2013 data)

In [30]:


statement = """SELECT * FROM {}.{}
    WHERE seq_id < {}
    ORDER BY seq_id""".format(
    database_name, table_name_parquet,train_seq_id
)

print(statement)

SELECT * FROM ccpp_aws_fp.final_data_parquet_v1
    WHERE seq_id < 5905
    ORDER BY seq_id


In [31]:
data_train = pd.read_sql(statement, conn)
data_train

/tmp/ipykernel_20296/3564220957.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_train = pd.read_sql(statement, conn)


,seq_id,ingestion_ts,gt_comp_dis_pressure,gt_exhaust_pressure,gt_inlet_temp,gt_air_filter_diff_pressure,solar_radiation,solar_energy,uv_index,gt_ambient_pressure,cloud_cover,sea_level_pressure,gt_energy_yield
0,1,2026-02-04 06:51:25.215,10.0720,18.132,1027.8,2.6356,346.2,1.2,3.0,1024.7,24.7,1027.3,100.02
1,2,2026-02-04 06:51:25.215,10.3860,19.480,1006.5,3.6384,33.8,0.1,0.0,1017.7,24.2,1028.0,100.03
2,3,2026-02-04 06:51:25.215,9.9408,17.982,1037.5,2.5758,761.2,2.7,8.0,1013.5,43.4,1017.8,100.04
3,4,2026-02-04 06:51:25.215,10.0930,18.091,1028.7,2.6372,543.4,2.0,5.0,1023.3,27.8,1027.1,100.07
4,5,2026-02-04 06:51:25.215,10.0720,18.232,1026.9,3.8423,0.0,0.0,0.0,1013.7,45.0,1021.9,100.14
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5899,5900,2026-02-04 06:51:25.215,11.6280,25.499,1074.8,2.9950,0.9,0.0,0.0,1017.2,55.7,1009.1,130.13
5900,5901,2026-02-04 06:51:25.215,11.5870,22.882,1073.6,3.1950,0.0,0.0,0.0,1033.3,52.3,1018.4,130.13
5901,5902,2026-02-04 06:51:25.215,11.6420,23.971,1076.3,3.4619,0.0,0.0,0.0,1012.3,0.0,1019.2,130.13
5902,5903,2026-02-04 06:51:25.215,11.5920,22.940,1072.4,2.9614,456.6,1.6,5.0,1008.1,87.9,992.2,130.13


### Load Validation Data 2014 January

In [35]:
statement = """SELECT * FROM {}.{}
    WHERE seq_id >= {} and seq_id < {}
    ORDER BY seq_id""".format(
    database_name, table_name_parquet,train_seq_id,val_seq_id
)

print(statement)

SELECT * FROM ccpp_aws_fp.final_data_parquet_v1
    WHERE seq_id >= 5905 and seq_id < 11810
    ORDER BY seq_id


In [36]:
data_val = pd.read_sql(statement, conn)
data_val

/tmp/ipykernel_20296/3443150230.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_val = pd.read_sql(statement, conn)


,seq_id,ingestion_ts,gt_comp_dis_pressure,gt_exhaust_pressure,gt_inlet_temp,gt_air_filter_diff_pressure,solar_radiation,solar_energy,uv_index,gt_ambient_pressure,cloud_cover,sea_level_pressure,gt_energy_yield
0,5905,2026-02-04 06:51:25.215,11.795,26.317,1070.5,3.1922,401.0,1.4,4.0,1002.9,23.9,1024.2,130.13
1,5906,2026-02-04 06:51:25.215,11.561,23.099,1072.6,3.0669,2.5,0.0,0.0,1008.5,89.1,999.3,130.13
2,5907,2026-02-04 06:51:25.215,11.714,22.995,1074.1,3.4150,0.0,0.0,0.0,1023.2,8.8,1028.2,130.14
3,5908,2026-02-04 06:51:25.215,12.017,26.025,1082.7,4.2980,2.6,0.0,0.0,1005.5,89.1,1008.0,130.14
4,5909,2026-02-04 06:51:25.215,11.619,28.618,1074.4,3.1668,223.1,0.8,2.0,1022.4,20.3,1013.2,130.14
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5900,11805,2026-02-04 06:51:25.215,13.237,30.768,1100.0,5.2301,353.4,1.3,4.0,1011.1,29.3,1010.8,147.96
5901,11806,2026-02-04 06:51:25.215,13.247,30.777,1099.9,4.9841,542.6,2.0,5.0,1005.5,85.5,1012.0,147.96
5902,11807,2026-02-04 06:51:25.215,13.239,30.058,1099.8,4.6899,0.0,0.0,0.0,1007.2,15.8,1009.9,147.96
5903,11808,2026-02-04 06:51:25.215,13.190,29.675,1100.0,4.2717,0.0,0.0,0.0,1011.3,24.0,1012.7,147.96


### Load Test Data 2014 February

In [37]:
statement = """SELECT * FROM {}.{}
    WHERE seq_id >= {} 
    ORDER BY seq_id""".format(
    database_name, table_name_parquet,val_seq_id
)

print(statement)

SELECT * FROM ccpp_aws_fp.final_data_parquet_v1
    WHERE seq_id >= 11810 
    ORDER BY seq_id


In [38]:
data_test = pd.read_sql(statement, conn)
data_test

/tmp/ipykernel_20296/2304454054.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_test = pd.read_sql(statement, conn)


,seq_id,ingestion_ts,gt_comp_dis_pressure,gt_exhaust_pressure,gt_inlet_temp,gt_air_filter_diff_pressure,solar_radiation,solar_energy,uv_index,gt_ambient_pressure,cloud_cover,sea_level_pressure,gt_energy_yield
0,11810,2026-02-04 06:51:25.215,13.206,29.560,1099.8,4.1605,0.0,0.0,0.0,1012.9,0.0,1015.0,147.96
1,11811,2026-02-04 06:51:25.215,13.181,30.379,1100.1,4.7453,747.3,2.7,7.0,1012.9,0.0,1009.7,147.97
2,11812,2026-02-04 06:51:25.215,13.198,29.797,1100.1,4.3650,787.9,2.8,8.0,1015.3,14.9,1012.9,147.97
3,11813,2026-02-04 06:51:25.215,13.233,30.344,1098.8,5.1058,170.9,0.6,2.0,1012.5,23.9,1012.7,147.98
4,11814,2026-02-04 06:51:25.215,13.246,30.035,1100.0,4.7132,683.5,2.5,7.0,1013.1,43.6,1016.0,147.98
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2950,14760,2026-02-04 06:51:25.215,15.039,36.573,1100.0,4.2813,0.0,0.0,0.0,1033.4,20.9,1038.3,177.49
2951,14761,2026-02-04 06:51:25.215,15.083,36.533,1100.0,4.3541,0.0,0.0,0.0,1034.1,89.1,1017.1,177.88
2952,14762,2026-02-04 06:51:25.215,15.029,37.105,1099.9,4.3915,0.0,0.0,0.0,1033.2,87.9,1015.3,177.91
2953,14763,2026-02-04 06:51:25.215,15.042,36.844,1100.0,4.3749,0.0,0.0,0.0,1034.0,89.1,1015.2,178.31


### Prepare the dataset

In [39]:
data_train = data_train.drop(['seq_id', 'ingestion_ts'], axis=1)
data_train

,gt_comp_dis_pressure,gt_exhaust_pressure,gt_inlet_temp,gt_air_filter_diff_pressure,solar_radiation,solar_energy,uv_index,gt_ambient_pressure,cloud_cover,sea_level_pressure,gt_energy_yield
0,10.0720,18.132,1027.8,2.6356,346.2,1.2,3.0,1024.7,24.7,1027.3,100.02
1,10.3860,19.480,1006.5,3.6384,33.8,0.1,0.0,1017.7,24.2,1028.0,100.03
2,9.9408,17.982,1037.5,2.5758,761.2,2.7,8.0,1013.5,43.4,1017.8,100.04
3,10.0930,18.091,1028.7,2.6372,543.4,2.0,5.0,1023.3,27.8,1027.1,100.07
4,10.0720,18.232,1026.9,3.8423,0.0,0.0,0.0,1013.7,45.0,1021.9,100.14
...,...,...,...,...,...,...,...,...,...,...,...
5899,11.6280,25.499,1074.8,2.9950,0.9,0.0,0.0,1017.2,55.7,1009.1,130.13
5900,11.5870,22.882,1073.6,3.1950,0.0,0.0,0.0,1033.3,52.3,1018.4,130.13
5901,11.6420,23.971,1076.3,3.4619,0.0,0.0,0.0,1012.3,0.0,1019.2,130.13
5902,11.5920,22.940,1072.4,2.9614,456.6,1.6,5.0,1008.1,87.9,992.2,130.13


In [40]:
data_val = data_val.drop(['seq_id', 'ingestion_ts'], axis=1)
data_val

,gt_comp_dis_pressure,gt_exhaust_pressure,gt_inlet_temp,gt_air_filter_diff_pressure,solar_radiation,solar_energy,uv_index,gt_ambient_pressure,cloud_cover,sea_level_pressure,gt_energy_yield
0,11.795,26.317,1070.5,3.1922,401.0,1.4,4.0,1002.9,23.9,1024.2,130.13
1,11.561,23.099,1072.6,3.0669,2.5,0.0,0.0,1008.5,89.1,999.3,130.13
2,11.714,22.995,1074.1,3.4150,0.0,0.0,0.0,1023.2,8.8,1028.2,130.14
3,12.017,26.025,1082.7,4.2980,2.6,0.0,0.0,1005.5,89.1,1008.0,130.14
4,11.619,28.618,1074.4,3.1668,223.1,0.8,2.0,1022.4,20.3,1013.2,130.14
...,...,...,...,...,...,...,...,...,...,...,...
5900,13.237,30.768,1100.0,5.2301,353.4,1.3,4.0,1011.1,29.3,1010.8,147.96
5901,13.247,30.777,1099.9,4.9841,542.6,2.0,5.0,1005.5,85.5,1012.0,147.96
5902,13.239,30.058,1099.8,4.6899,0.0,0.0,0.0,1007.2,15.8,1009.9,147.96
5903,13.190,29.675,1100.0,4.2717,0.0,0.0,0.0,1011.3,24.0,1012.7,147.96


In [41]:
data_test = data_test.drop(['seq_id', 'ingestion_ts'], axis=1)
data_test

,gt_comp_dis_pressure,gt_exhaust_pressure,gt_inlet_temp,gt_air_filter_diff_pressure,solar_radiation,solar_energy,uv_index,gt_ambient_pressure,cloud_cover,sea_level_pressure,gt_energy_yield
0,13.206,29.560,1099.8,4.1605,0.0,0.0,0.0,1012.9,0.0,1015.0,147.96
1,13.181,30.379,1100.1,4.7453,747.3,2.7,7.0,1012.9,0.0,1009.7,147.97
2,13.198,29.797,1100.1,4.3650,787.9,2.8,8.0,1015.3,14.9,1012.9,147.97
3,13.233,30.344,1098.8,5.1058,170.9,0.6,2.0,1012.5,23.9,1012.7,147.98
4,13.246,30.035,1100.0,4.7132,683.5,2.5,7.0,1013.1,43.6,1016.0,147.98
...,...,...,...,...,...,...,...,...,...,...,...
2950,15.039,36.573,1100.0,4.2813,0.0,0.0,0.0,1033.4,20.9,1038.3,177.49
2951,15.083,36.533,1100.0,4.3541,0.0,0.0,0.0,1034.1,89.1,1017.1,177.88
2952,15.029,37.105,1099.9,4.3915,0.0,0.0,0.0,1033.2,87.9,1015.3,177.91
2953,15.042,36.844,1100.0,4.3749,0.0,0.0,0.0,1034.0,89.1,1015.2,178.31


In [42]:
data_batch = data_test.drop(['gt_energy_yield'], axis=1)
data_batch

,gt_comp_dis_pressure,gt_exhaust_pressure,gt_inlet_temp,gt_air_filter_diff_pressure,solar_radiation,solar_energy,uv_index,gt_ambient_pressure,cloud_cover,sea_level_pressure
0,13.206,29.560,1099.8,4.1605,0.0,0.0,0.0,1012.9,0.0,1015.0
1,13.181,30.379,1100.1,4.7453,747.3,2.7,7.0,1012.9,0.0,1009.7
2,13.198,29.797,1100.1,4.3650,787.9,2.8,8.0,1015.3,14.9,1012.9
3,13.233,30.344,1098.8,5.1058,170.9,0.6,2.0,1012.5,23.9,1012.7
4,13.246,30.035,1100.0,4.7132,683.5,2.5,7.0,1013.1,43.6,1016.0
...,...,...,...,...,...,...,...,...,...,...
2950,15.039,36.573,1100.0,4.2813,0.0,0.0,0.0,1033.4,20.9,1038.3
2951,15.083,36.533,1100.0,4.3541,0.0,0.0,0.0,1034.1,89.1,1017.1
2952,15.029,37.105,1099.9,4.3915,0.0,0.0,0.0,1033.2,87.9,1015.3
2953,15.042,36.844,1100.0,4.3749,0.0,0.0,0.0,1034.0,89.1,1015.2


In [43]:
prefix = "CCPP-enery-prediction-linear-regression"

In [44]:
train_file = "train_data.csv"
data_train.to_csv(train_file, index=False, header=False)
sess.upload_data(train_file, key_prefix="{}/train".format(prefix))

validation_file = "validation_data.csv"
data_val.to_csv(validation_file, index=False, header=False)
sess.upload_data(validation_file, key_prefix="{}/validation".format(prefix))

test_file = "test_data.csv"
data_test.to_csv(test_file, index=False, header=False)
sess.upload_data(test_file, key_prefix="{}/batch".format(prefix))

batch_file = "batch_data.csv"
data_batch.to_csv(batch_file, index=False, header=False)
sess.upload_data(batch_file, key_prefix="{}/batch".format(prefix))

's3://sagemaker-us-east-1-483525480215/CCPP-enery-prediction-linear-regression/batch/batch_data.csv'

### Model Training Setup and Training

In [47]:
%%time
from time import gmtime, strftime

job_name = "lr-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
output_location = "s3://{}/{}/output/{}".format(bucket, prefix, job_name)
image = sagemaker.image_uris.retrieve(
    framework="linear-learner",   # built-in algorithm for regression/classification
    region=boto3.Session().region_name,
    version="latest"               # always safe to use "latest"
)

linear_estimator = sagemaker.estimator.Estimator(
    image,
    role,
    instance_count=1,
    instance_type="ml.m5.large",
    volume_size=50,
    input_mode="File",
    output_path=output_location,
    sagemaker_session=sess,
)
linear_estimator.set_hyperparameters(
    predictor_type="regressor",   # must set this for regression
    mini_batch_size=100,          # batch size for training
    epochs=15,                    # number of passes over the dataset
    learning_rate=0.01,           # similar to eta
    optimizer="adam"              # can also be 'sgd', 'ftrl'
)
train_data = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/train".format(bucket, prefix),
    distribution="FullyReplicated",
    content_type="text/csv",
    s3_data_type="S3Prefix",
)
validation_data = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/validation".format(bucket, prefix),
    distribution="FullyReplicated",
    content_type="text/csv",
    s3_data_type="S3Prefix",
)
data_channels = {"train": train_data, "validation": validation_data}

# Start training by calling the fit method in the estimator
linear_estimator.fit(inputs=data_channels, job_name=job_name, logs=True)


INFO:sagemaker.image_uris:Same images used for training and inference. Defaulting to image scope: inference.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: lr-2026-02-04-07-41-14


2026-02-04 07:41:16 Starting - Starting the training job...
2026-02-04 07:41:30 Starting - Preparing the instances for training...
2026-02-04 07:41:53 Downloading - Downloading input data...
2026-02-04 07:42:33 Downloading - Downloading the training image.........
2026-02-04 07:43:59 Training - Training image download completed. Training in progress.Docker entrypoint called with argument(s): train
Running default environment configuration script
[02/04/2026 07:44:04 INFO 140550108096320] Reading default configuration from /opt/amazon/lib/python3.8/site-packages/algorithm/resources/default-input.json: {'mini_batch_size': '1000', 'epochs': '15', 'feature_dim': 'auto', 'use_bias': 'true', 'binary_classifier_model_selection_criteria': 'accuracy', 'f_beta': '1.0', 'target_recall': '0.8', 'target_precision': '0.8', 'num_models': 'auto', 'num_calibration_samples': '10000000', 'init_method': 'uniform', 'init_scale': '0.07', 'init_sigma': '0.01', 'init_bias': '0.0', 'optimizer': 'auto', 'loss':

### Setup Batch Transfomer Estimator

In [48]:
%%time

linear_transformer = linear_estimator.transformer(1, "ml.m4.xlarge")

# start a transform job
input_location = "s3://{}/{}/batch/{}".format(
    bucket, prefix, batch_file
)  # use input data without ID column
linear_transformer.transform(input_location, content_type="text/csv", split_type="Line")
linear_transformer.wait()

INFO:sagemaker:Creating model with name: linear-learner-2026-02-04-07-45-33-219
INFO:sagemaker:Creating transform job with name: linear-learner-2026-02-04-07-45-33-934


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ /opt/conda/lib/python3.12/site-packages/IPython/core/magics/execution.py:1370 in time            │
│                                                                                                  │
│   1367 │   │   else:                                                                             │
│   1368 │   │   │   st = clock2()                                                                 │
│   1369 │   │   │   try:                                                                          │
│ ❱ 1370 │   │   │   │   exec(code, glob, local_ns)                                                │
│   1371 │   │   │   │   out = None                                                                │
│   1372 │   │   │   │   # multi-line %%time case                                                  │
│   1373 │   │   │   │   if expr_val is not None:                                                  │
│ in <module>:7                                                                                    │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:346 in wrapper    │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/transformer.py:302 in transform                │
│                                                                                                  │
│   299 │   │   │   sagemaker_session=self.sagemaker_session,                                      │
│   300 │   │   )                                                                                  │
│   301 │   │                                                                                      │
│ ❱ 302 │   │   self.latest_transform_job = _TransformJob.start_new(                               │
│   303 │   │   │   self,                                                                          │
│   304 │   │   │   data,                                                                          │
│   305 │   │   │   data_type,                                                                     │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/transformer.py:636 in start_new                │
│                                                                                                  │
│   633 │   │   │   batch_data_capture_config,                                                     │
│   634 │   │   )                                                                                  │
│   635 │   │                                                                                      │
│ ❱ 636 │   │   transformer.sagemaker_session.transform(**transform_args)                          │
│   637 │   │                                                                                      │
│   638 │   │   return cls(transformer.sagemaker_session, tra

### Plot the Batch Transform Result Compare to Ground Truth

In [49]:
import re


def get_csv_output_from_s3(s3uri, batch_file):
    file_name = "{}.out".format(batch_file)
    match = re.match("s3://([^/]+)/(.*)", "{}/{}".format(s3uri, file_name))
    output_bucket, output_prefix = match.group(1), match.group(2)
    s3.download_file(output_bucket, output_prefix, file_name)
    return pd.read_csv(file_name, sep=",", header=None)

In [50]:
s3 = boto3.client("s3")

In [51]:
output_df = get_csv_output_from_s3(linear_transformer.output_path, batch_file)
output_df.head(8)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 output_df = get_csv_output_from_s3(sm_transformer.output_path, batch_file)                   │
│   2 output_df.head(8)                                                                            │
│   3                                                                                              │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
NameError: name 'sm_transformer' is not defined

In [31]:
output_df

,0
0,0.168284
1,0.168284
2,0.168284
3,0.168284
4,0.168284
...,...
644,0.262286
645,0.262286
646,0.262286
647,0.262286


In [ ]:
compare_df = data_test[['gt_energy_yield']]

In [ ]:
compare_df

In [ ]:
combined_df = pd.concat([compare_df, output_df], axis=1)

In [ ]:
combined_df

In [ ]:
combined_df.rename(columns={0: 'Predited Energy'}, inplace=True)

In [ ]:
combined_df

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))

# Plotting the first column with a solid line
plt.plot(combined_df.index, combined_df['kwh'], linestyle='-', color='blue', label='Column 1 - Solid')

# Plotting the second column with a dotted line
plt.plot(combined_df.index, combined_df['Predited Energy'], linestyle=':', color='red', label='Column 2 - Dotted')
plt.legend()
# Show plot
plt.show()

In [28]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>